In [1]:
pip install groq python-dotenv

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 138.3/138.3 kB 9.0 MB/s eta 0:00:00


In [3]:
GROQ_API_KEY="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"

In [15]:
import os
from dotenv import load_dotenv
from groq import Groq

# Load environment variables (if any .env file exists)
load_dotenv()
client = Groq(api_key=GROQ_API_KEY)

BASE_MODEL = "llama-3.1-8b-instant"

DEFINE EXPERT CONFIGS

In [7]:
MODEL_CONFIG = {
    "technical": {
        "system_prompt": """You are a Technical Support Expert.
You are rigorous, precise, and code-focused.
Provide debugging steps, explain errors clearly, and give corrected code snippets when needed.
Be concise but technically accurate."""
    },
    "billing": {
        "system_prompt": """You are a Billing Support Specialist.
You are empathetic, policy-driven, and professional.
Help users with refunds, subscription issues, and charges.
Always acknowledge the user's concern before explaining policies."""
    },
    "general": {
        "system_prompt": """You are a friendly and helpful customer support assistant.
Answer general questions clearly and politely."""
    }
}

ROUTER FUNCTION

In [8]:
def route_prompt(user_input: str) -> str:
    """
    Uses LLM to classify the user's query into:
    technical, billing, general, or tool
    Returns ONLY the category name.
    """

    routing_prompt = f"""
Classify the following user query into one of these categories:
[technical, billing, general, tool]

Return ONLY the category name. No explanation.

User Query:
{user_input}
"""

    response = client.chat.completions.create(
        model=BASE_MODEL,
        temperature=0,  # deterministic
        messages=[
            {"role": "system", "content": "You are a strict intent classifier."},
            {"role": "user", "content": routing_prompt}
        ]
    )

    category = response.choices[0].message.content.strip().lower()
    return category


BONUS TOOL EXPERT

In [9]:
def get_bitcoin_price():
    """
    Mock function for fetching Bitcoin price.
    (In real world, call a crypto API.)
    """
    return "The current price of Bitcoin is $62,450 (mock data)."

ORCHESTRATOR

In [10]:
def process_request(user_input: str) -> str:
    """
    1. Route the query
    2. Load appropriate expert
    3. Generate response
    """

    category = route_prompt(user_input)

    print(f"[Router Decision]: {category}")

    # Handle Tool Expert separately
    if category == "tool":
        if "bitcoin" in user_input.lower():
            return get_bitcoin_price()
        else:
            return "Tool requested, but no tool available."

    # Fallback safety
    if category not in MODEL_CONFIG:
        category = "general"

    system_prompt = MODEL_CONFIG[category]["system_prompt"]

    response = client.chat.completions.create(
        model=BASE_MODEL,
        temperature=0.7,  # more creative for experts
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_input}
        ]
    )

    return response.choices[0].message.content

TESTING

In [17]:
if __name__ == "__main__":

    while True:
        user_query = input("\nEnter your query (type 'exit' to quit): ")

        if user_query.lower() == "exit":
            break

        result = process_request(user_query)
        print("\nResponse:\n", result)


Enter your query (type 'exit' to quit): i was charged twice for my subscription this month
[Router Decision]: billing

Response:
 I'm so sorry to hear that you've been charged twice for your subscription this month. That can be really frustrating and confusing. I'm here to help you understand what might have happened and see if we can resolve the issue for you.

Can you please confirm your account information and subscription details with me? This will help me look into the matter further. What's your subscription plan and what date did you notice the duplicate charge?

Enter your query (type 'exit' to quit): exit
